In [ ]:
#!pip install openpyxl

import pandas as pd
import requests
from auxiliares import funcoes

In [ ]:
url = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
response = requests.get(url)
print('Conectando...')

if response.status_code == 200:
    print('Conectado!')
    dados = response.json()
    lista_cidades = []
    
    for mun in dados:

        micro = mun.get('microrregiao') or {}
        meso = micro.get('mesorregiao') or {}
        uf = meso.get('UF') or {}
        regiao = uf.get('regiao') or {}
        
        # Extraindo a árvore nova
        imediata = mun.get('regiao-imediata') or {}
        intermediaria = imediata.get('regiao-intermediaria') or {}
        
        lista_cidades.append({
            'Pais': 'Brasil',
            'Região': regiao.get('nome'),
            'Estado': uf.get('nome'),
            'Sigla_estado': uf.get('sigla'),
            'Mesoregião': meso.get('nome'),
            'Região_intermediaria': intermediaria.get('nome'),
            'Microregião': micro.get('nome'),
            'Região_imediata': imediata.get('nome'),
            'Municipios': mun.get('nome')
        })

    # Criação do DataFrame Gabarito
    df_gabarito = pd.DataFrame(lista_cidades)
    
    # Aplicando a padronização
    def limpar_texto_gabarito(coluna):
        return coluna.astype(str).str.upper().str.strip().str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
    
    colunas_texto = ['Região', 'Estado', 'Mesoregião', 'Região_intermediaria', 'Microregião', 'Região_imediata', 'Municipios']
    for col in colunas_texto:
        df_gabarito[col] = limpar_texto_gabarito(df_gabarito[col])

    print("Processado!")
    display(df_gabarito.head())

else:
    print(f"Erro ao acessar IBGE: Status {response.status_code}")

In [ ]:
funcoes.analise_dados(df_gabarito)

In [ ]:
mascara_nulos = (df_gabarito == 'NONE').any(axis=1) | df_gabarito.isna().any(axis=1)
display(df_gabarito[mascara_nulos])

In [ ]:
# 1. Cria a máscara para localizar a linha exata do município novo
mascara_boa = df_gabarito['Municipios'] == 'BOA ESPERANCA DO NORTE'

# 2. Preenche os buracos com a hierarquia da região de Sorriso/MT
df_gabarito.loc[mascara_boa, 'Região'] = 'CENTRO OESTE'
df_gabarito.loc[mascara_boa, 'Estado'] = 'MATO GROSSO'
df_gabarito.loc[mascara_boa, 'Sigla_estado'] = 'MT'
df_gabarito.loc[mascara_boa, 'Mesoregião'] = 'NORTE MATO-GROSSENSE'
df_gabarito.loc[mascara_boa, 'Região_intermediaria'] = 'SINOP'
df_gabarito.loc[mascara_boa, 'Microregião'] = 'ALTO TELES PIRES'
df_gabarito.loc[mascara_boa, 'Região_imediata'] = 'SORRISO'

print("✅ Dados de Boa Esperança do Norte corrigidos:")
display(df_gabarito[mascara_boa])

In [ ]:
funcoes.contagem_valores(df_gabarito)

In [12]:
escolha = int(input('Escolha o método de salvamento: 1 - parquet | 2 - CSV | 3 - EXCEL'))
if escolha == 1:
    df_gabarito.to_parquet('dicionario_municipios_ibge.parquet', index=False)
    print("✅ Arquivo Parquet salvo com sucesso!")    
elif escolha == 2:
    df_gabarito.to_csv('dicionario_municipios_ibge.csv', index=False, encoding='utf-8-sig')
    print("✅ Arquivo CSV salvo com sucesso!")
elif escolha == 3:
    df_gabarito.to_excel('dicionario_municipios_ibge.xlsx', index=False)
    print("✅ Arquivo Excel salvo com sucesso!")   
else:
    print("⚠️ Opção inválida.")
print("Arquivo criado!")

✅ Arquivo Excel salvo com sucesso!
Arquivo criado!
